# Electrophysiology Tutorial 5: Eikonal Models


<img src="../eikonal-wave.gif" width="50%">


This tutorial shows how to solve Eikonal models and how to recover transemembrane
potential fields and use them to calculate the ECG corresponding to the activation pattern.

> **Todo**
>
> Provide context.

# Introduction
Depolarization can be modeled spatially as a wave propagating through cardiac tissue.
The arrival times of this wave is the solution of the Eikonal equation in the Reaction-Eikonal
split [NeicCamposPrassl:2017:ECE](@citet):
$$
\begin{aligned}
  C_{\textrm{m}} \partial_t \varphi &=  I_{\textrm{foot}}(t_{\textrm{arrival}}, t) - I_{\textrm{ion}}(\varphi, \boldsymbol{s}, t) & \textrm{pointwise in} \: \Omega \, , \\
  \partial_t \boldsymbol{s} &= \mathbf{g}(\varphi, \boldsymbol{s}) & \textrm{pointwise in}  \: \Omega \, , \\
  I_{\textrm{foot}}(t_{\textrm{arrival}}, t) &= -\frac{A}{\tau}\cdot e^{\frac{t - t_{\textrm{arrival}}}{\tau}} & \textrm{pointwise in}  \: \Omega \;\,
\end{aligned}
$$
Where $C_{\textrm{m}}$ is the membrane capacitance, $\varphi$ is the transmembrane
potential, $I_{\textrm{foot}}$ is the stimulating current, $I_{\textrm{ion}}$ is
the ionic cureent resulting from the cell model. $\boldsymbol{s}$ is the state vector
for the cell model, and $\mathbf{g}$ is the function evolving the cell model states in time.

Wave arrival times are obtained by solving the Eikonal equation:
$$
\sqrt{\nabla t_{\textrm{arrival}}^T \boldsymbol{V} \nabla t_{\textrm{arrival}}} = 1 \qquad \textrm{in}  \: \Omega
$$

Where $\boldsymbol{V}$ the conduction velocity, which was shown to be proportional to
the square root of the diffusivities of the cardiac tissue [Weingart:1977:AOIC](@citet) (That's the earliest mention I
could find but I remember there was an older one in the 50s).


## Commented Program

In [1]:
using Thunderbolt, LinearAlgebra, StaticArrays, OrdinaryDiffEqRosenbrock, FastIterativeMethod

> **Todo**
>
> The initializer API is not yet finished and hence we deconstruct stuff here manually.
> Please note that this method is quite fragile w.r.t. to many changes you can make in the code below.

In [2]:
function steady_state_initializer!(u₀, f::Thunderbolt.ReactionEikonalFunction)
    # TODO cleaner implementation. We need to extract this from the types or via dispatch.
    odefun = f.ode_function
    ionic_model = odefun.cellmodel

    φ₀ = @view u₀[1:solution_size(f.eikonal_function)]
    # TODO extraction these via utility functions
    s₀flat = @view u₀[(solution_size(f.eikonal_function) + 1):end]
    # Should not be reshape but some array of arrays fun
    s₀ = reshape(
        s₀flat, (solution_size(f.eikonal_function), Thunderbolt.num_states(ionic_model) - 1)
    )
    default_values = Thunderbolt.default_initial_state(ionic_model)

    φ₀ .= default_values[1]
    for i in 1:(Thunderbolt.num_states(ionic_model) - 1)
        s₀[:, i] .= default_values[i + 1]
    end
    return
end

steady_state_initializer! (generic function with 1 method)

We start by generating a tetrahedral mesh, as the fast iterative method solver
accepts tetrahedral meshes only.

In [3]:
heart_mesh = generate_ideal_lv_mesh(
    12, 2, 5;
    inner_radius = 0.7,
    outer_radius = 1.0,
    longitudinal_upper = 0.0,
    apex_inner = 0.7,
    apex_outer = 1.0
) |> Thunderbolt.hexahedralize |> Thunderbolt.tetrahedralize;

heart_mesh = Thunderbolt.to_mesh(heart_mesh.grid)

Thunderbolt.SimpleMesh{3, Tetrahedron, Float64} with 13248 Tetrahedron cells and 2549 nodes
  Surface subdomains:
    Epicardium 552 Tetrahedron cells
    Base 192 Tetrahedron cells
    Endocardium 552 Tetrahedron cells


> **Tip**
>
> We can also load realistic geometries with external formats. For this simply use either FerriteGmsh.jl
> or one of the loader functions stated in the mesh API.

We also generate a coordinate system to be used for checking points for initial activation.

In [4]:
cs = compute_lv_coordinate_system(heart_mesh)

LVCoordinateSystem{Float64, DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}, LagrangeCollection{1}}(DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}(SubDofHandler{DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}}[SubDofHandler{DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}}(DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}(#= circular reference @-3 =#), OrderedCollections.OrderedSet{Int64}([1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  13239, 13240, 13241, 13242, 13243, 13244, 13245, 13246, 13247, 13248]), [:coordinates], Interpolation[Lagrange{RefTetrahedron, 1}()], Int64[], 4)], [:coordinates], [1, 2, 3, 4, 1, 5, 2, 4, 1, 6  …  2436, 2549, 2547, 322, 543, 2549, 2547, 2436, 322, 2549], [1, 5, 9, 13, 17, 21, 25, 29, 33, 37  …  52953, 52957, 52961, 52965, 52969, 52973, 52977, 52981, 52985, 52989], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  1, 1, 1, 1, 1, 1, 1, 1, 1, 1], true, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}(Grid{3, Tetra

For this tutorial we define a uniform endocardial activation

In [5]:
activation_protocol = Thunderbolt.UniformEndocardialActivationProtocol(Dict{String, Float64}(),
cs
)

Thunderbolt.UniformEndocardialActivationProtocol{LVCoordinateSystem{Float64, DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}, LagrangeCollection{1}}}(Dict{String, Float64}(), LVCoordinateSystem{Float64, DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}, LagrangeCollection{1}}(DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}(SubDofHandler{DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}}[SubDofHandler{DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}}(DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}(#= circular reference @-3 =#), OrderedCollections.OrderedSet{Int64}([1, 2, 3, 4, 5, 6, 7, 8, 9, 10  …  13239, 13240, 13241, 13242, 13243, 13244, 13245, 13246, 13247, 13248]), [:coordinates], Interpolation[Lagrange{RefTetrahedron, 1}()], Int64[], 4)], [:coordinates], [1, 2, 3, 4, 1, 5, 2, 4, 1, 6  …  2436, 2549, 2547, 322, 543, 2549, 2547, 2436, 322, 2549], [1, 5, 9, 13, 17, 21, 25, 29, 33, 37  …  52953, 52957,

For our toy problem we use a very simple microstructure.

In [6]:
microstructure = OrthotropicMicrostructureModel(
    ConstantCoefficient((Vec(0.0, 0.0, 1.0))),
    ConstantCoefficient((Vec(0.0, 1.0, 0.0))),
    ConstantCoefficient((Vec(1.0, 0.0, 0.0)))
)

OrthotropicMicrostructureModel{ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}}(ConstantCoefficient{Vec{3, Float64}}([0.0, 0.0, 1.0]), ConstantCoefficient{Vec{3, Float64}}([0.0, 1.0, 0.0]), ConstantCoefficient{Vec{3, Float64}}([1.0, 0.0, 0.0]))

With the microstructure we setup the diffusion tensor field in spectral form.
> **Todo**
>
> citation

In [7]:
κ₁ = 0.17 * 0.62 / (0.17 + 0.62)
κᵣ = 0.019 * 0.24 / (0.019 + 0.24)
diffusion_tensor_field = SpectralTensorCoefficient(
    microstructure,
    ConstantCoefficient(SVector(κ₁, κ₁, κ₁))
)

SpectralTensorCoefficient{OrthotropicMicrostructureModel{ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}}, ConstantCoefficient{StaticArraysCore.SVector{3, Float64}}}(OrthotropicMicrostructureModel{ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}}(ConstantCoefficient{Vec{3, Float64}}([0.0, 0.0, 1.0]), ConstantCoefficient{Vec{3, Float64}}([0.0, 1.0, 0.0]), ConstantCoefficient{Vec{3, Float64}}([1.0, 0.0, 0.0])), ConstantCoefficient{StaticArraysCore.SVector{3, Float64}}([0.13341772151898734, 0.13341772151898734, 0.13341772151898734]))

Now we choose our *inner* cell model, used to compute the ionic currents, since
the eikonal-based foot current acts as a wrapper around ionic cell models.

In [8]:
cellmodel = Thunderbolt.PCG2019()

Thunderbolt.ParametrizedPCG2019Model{Float64}(12.0, -52.244, 6.5472, 0.12, -78.7, 5.93, 0.799163, 6.80738, 0.73893, -91.9655, 12.4997, 0.1688, 14.3116, 11.462, -47.9286, 4.9314, 9.90669, 0.11503, 0.7, 4.3, -15.7, 4.6, 30.0, 0.056, -26.6, 6.5, 334.0, -49.6, 23.5, 0.008, 24.6, 12.1, 628.0, 65.0, -85.0, 50.0)

We define the monodomain parameters to be used in the split.

In [9]:
heart_model = MonodomainModel(
    ConstantCoefficient(1.0),
    ConstantCoefficient(1.0),
    diffusion_tensor_field,
    activation_protocol,
    cellmodel,
    :φₘ, :s
)

MonodomainModel{ConstantCoefficient{Float64}, ConstantCoefficient{Float64}, SpectralTensorCoefficient{OrthotropicMicrostructureModel{ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}}, ConstantCoefficient{StaticArraysCore.SVector{3, Float64}}}, Thunderbolt.UniformEndocardialActivationProtocol{LVCoordinateSystem{Float64, DofHandler{3, Thunderbolt.SimpleMesh{3, Tetrahedron, Float64}}, LagrangeCollection{1}}}, Thunderbolt.ParametrizedPCG2019Model{Float64}}(ConstantCoefficient{Float64}(1.0), ConstantCoefficient{Float64}(1.0), SpectralTensorCoefficient{OrthotropicMicrostructureModel{ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}}, ConstantCoefficient{StaticArraysCore.SVector{3, Float64}}}(OrthotropicMicrostructureModel{ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}}(ConstantCoefficient{Vec{3, Float6

We semidiscretize the using a reaction-diffusion split and provide the desired activation
protocol. Here, we activate the full indocardium defined as nodes with less than 0.05
transmural coordinate in the provided coordinate system by default.

In [10]:
heart_odeform = semidiscretize(
    ReactionEikonalSplit(heart_model, cs),
    (
        FiniteElementDiscretization(Dict(:φₘ => LagrangeCollection{1}())),
        Thunderbolt.SimplicialEikonalDiscretization(;
            activation_protocol,
            subdomains = String[]
        ),
    ),
    heart_mesh
)

ReactionEikonalFunction{Thunderbolt.var"#semidiscretize##12#semidiscretize##13"{Vector{Float64}, Thunderbolt.ParametrizedPCG2019Model{Float64}}, Thunderbolt.EikonalFunction{Float64, Vector{Vec{3, Float64}}, Vector{NTuple{4, Int64}}, Ferrite.CollectionsOfViews.ArrayOfVectorViews{Int64, 1}, SpectralTensorCoefficient{OrthotropicMicrostructureModel{ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}, ConstantCoefficient{Vec{3, Float64}}}, ConstantCoefficient{StaticArraysCore.SVector{3, Float64}}}, OrderedCollections.OrderedSet{Int64}}}(Thunderbolt.var"#semidiscretize##12#semidiscretize##13"{Vector{Float64}, Thunderbolt.ParametrizedPCG2019Model{Float64}}([Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf  …  Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf, Inf], Thunderbolt.ParametrizedPCG2019Model{Float64}(12.0, -52.244, 6.5472, 0.12, -78.7, 5.93, 0.799163, 6.80738, 0.73893, -91.9655, 12.4997, 0.1688, 14.3116, 11.462, -47.9286, 4.9314, 9.90669, 0.11503, 0.7, 4.3, -15.7, 4

We allocate the solution vector for the reaction part. The activation times vector is allocated internally.

In [11]:
u₀ = zeros(Float64, Thunderbolt.num_states(heart_odeform.ode_function.cellmodel) * length(heart_mesh.grid.nodes))

17843-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

Apply the initial conditions.

In [12]:
steady_state_initializer!(u₀, heart_odeform)

Here we solve for the wave arrival time using the package *Eikonal.jl* as it's
required to be solved for once before timestepping.

In [13]:
FastIterativeMethod.solve!(heart_odeform, heart_mesh)

2549-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 1.955547294595218
 1.9576203033639725
 1.9558353686194496
 1.3964852205371976
 1.3979366378894262
 1.3966980715420516
 1.9555472945952175
 1.9576203033639725
 1.9558353686194496

> **Danger**
>
> The fast iterative method solver in `Eikonal.jl` uses `@inbounds` a lot
> in its internals. It was observed that forcing bounds checks results in an
> enormous amount of dynamic allocations that can easily result in running out of memory.

In [14]:
nodal_timings = heart_odeform.ode_function.activation_timings
VTKGridFile(("ep05_eikonal_timings.vtu"), heart_mesh.grid) do vtk          # hide
    vtk.vtk["timings"] = nodal_timings    # hide
end                                                                                 # hide

VTKGridFile for the closed file "ep05_eikonal_timings.vtu".

Now we solve for the reaction part, where we evolve state variables and transmembrane potential.

In [15]:
dt₀ = 0.01
dtvis = 0.5
Tₘₐₓ = 50.0
Tₘₐₓ = dtvis # hide
tspan = (0.0, Tₘₐₓ)

single_prob = ODEProblem(
    (du, u, m, t) -> Thunderbolt.cell_rhs!(du, u, m.stim_offset, t, m),
    Thunderbolt.default_initial_state(cellmodel),
    (first(tspan), last(tspan)),
    Thunderbolt.StimulatedCellModel(; cell_model = cellmodel),
)
problem = SciMLBase.EnsembleProblem(single_prob, prob_func = heart_odeform.ode_function);

sim = solve(problem, Rodas5P(), saveat = collect(0.0:dtvis:Tₘₐₓ), trajectories = length(heart_odeform.ode_function.activation_timings))
dh = cs.dh |> deepcopy
Thunderbolt.reorder_nodal!(dh)

io = ParaViewWriter("ep05_eikonal")

φₘfield = copy(nodal_timings)           # hide
for t in 0.0:dtvis:Tₘₐₓ                                                          # hide
    for i in 1:length(φₘfield)                                                  # hide
        φₘfield[i] = sim.u[i](t)[1]                                             # hide
    end                                                                         # hide
    store_timestep!(io, t, dh.grid) do file                                     # hide
        Thunderbolt.store_timestep_field!(file, t, dh, φₘfield, :coordinates)   # hide
    end                                                                         # hide
end                                                                             # hide

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*